<a href="https://colab.research.google.com/github/lubbad-aljazeera/paid-activity-data-pipeline/blob/main/META_Data_Pipeline_Extend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install db-dtypes
!pip install google-cloud-bigquery
!pip install google-cloud-storage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.0/262.0 kB 9.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
from google.cloud import storage, bigquery
from google.colab import auth
import re
import io
import numpy as np
import csv # Import the csv module for quoting options
from datetime import datetime # Import datetime for backup table naming


auth.authenticate_user()
print("✅ Google Cloud authenticated successfully.")

bucket_name = "aj360_data"
dataset_id = "aj360"
table_id = "meta_ads_performance" # Keep this table_id for the META_ADS data
# x_ads_table_id = "x_ads_performance" # New table_id for X_ADS data
project_id = "ajgc-dig-jwc-prod-jawacloud-01"
dataset_ref = bigquery.DatasetReference(project_id, dataset_id)
meta_ads_table_ref = dataset_ref.table(table_id) # Table reference for META_ADS
# x_ads_table_ref = dataset_ref.table(x_ads_table_id) # Table reference for X_ADS

clean_file = "meta_ads_clean.csv" # Keep clean file name for META_ADS
# x_ads_clean_file = "x_ads_clean.csv" # New clean file name for X_ADS

# Process META_ADS data (existing logic)
prefix = "META_ADS/meta_ads_performance_5-7Mar2026.csv"
print(f"Listing CSV files in bucket: {bucket_name} with prefix: {prefix}")
storage_client = storage.Client(project=project_id)
bucket = storage_client.bucket(bucket_name)
blobs = list(bucket.list_blobs(prefix=prefix))
csv_files = [blob.name for blob in blobs if blob.name.endswith('.csv')]

if not csv_files:
    print("❌ No CSV files found for META_ADS.")
else:
    print(f"✅ Found {len(csv_files)} CSV files to process for META_ADS.")

    bq_client = bigquery.Client(project=project_id)
    try:
        bq_client.get_dataset(dataset_ref)
        print(f"✅ Dataset {dataset_id} exists.")
    except:
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US"
        dataset = bq_client.create_dataset(dataset)
        print(f"✅ Created dataset {dataset_id}.")

    print("Loading lookup table from BigQuery...")
    term_lookup = {} # Initialize term_lookup
    try:
        lookup_df = bq_client.query("""
            SELECT term, show_title_ar, show_title_en
            FROM `ajgc-dig-jwc-prod-jawacloud-01.aj360.paid_ads_shows_lookup`
        """).to_dataframe()
        term_lookup = dict(zip(lookup_df['term'], lookup_df['show_title_en']))
        print(f"✅ Loaded lookup table with {len(term_lookup)} terms.")
    except Exception as e:
        print(f"❌ Failed to load lookup table: {e}")
        csv_files = [] # Clear csv_files if lookup fails to prevent processing with incomplete data

    unified_schema = {}
    schema = None

    if csv_files: # Only proceed with schema inference if there are files to process
        print("\n--- Schema Inference (META_ADS) ---")
        for file in csv_files[:5]: # Infer schema from a sample of files
            try:
                print(f"Inferring schema from: {file}")
                blob = bucket.blob(file)
                # Use pandas to read a small portion to infer types more accurately
                with blob.open("r", encoding='utf-8', errors='ignore') as f:
                     df_temp = pd.read_csv(f, nrows=1000, on_bad_lines='skip')


                if df_temp.empty:
                    print(f"⚠️ Empty file or failed to read for schema inference: {file}. Skipping.")
                    continue

                df_temp.columns = [re.sub(r'[^a-zA-Z0-9_ /]', '', col).lower() for col in df_temp.columns]
                df_temp.columns = [col.replace('/', '_') for col in df_temp.columns]
                df_temp.columns = [col.replace(' ', '_') for col in df_temp.columns]

                # Add 'show_title' to schema consideration if 'ad_name' exists
                if 'ad_name' in df_temp.columns:
                    if 'show_title' not in unified_schema:
                        unified_schema['show_title'] = 'STRING'


                for col, dtype in df_temp.dtypes.items():
                    print(f"  {col}: {dtype}")
                    # Map pandas dtypes to BigQuery dtypes
                    if dtype == 'object':
                        bq_type = 'STRING'
                    elif 'int' in str(dtype):
                        bq_type = 'INTEGER'
                    elif 'float' in str(dtype):
                        if col != 'clicks_all' and col != 'views' and col != 'unique_clicks_all':
                          bq_type = 'FLOAT'
                        else:
                          bq_type = 'INTEGER'
                    elif 'bool' in str(dtype):
                        bq_type = 'BOOLEAN'
                    elif 'datetime' in str(dtype):
                        bq_type = 'TIMESTAMP'
                    else:
                        bq_type = 'STRING' # Default to STRING for unknown types

                    # Use the most general type seen so far for each column
                    if col not in unified_schema:
                        unified_schema[col] = bq_type
                    else:
                        # Simple type promotion logic
                        if unified_schema[col] == 'FLOAT' and bq_type == 'FLOAT':
                            unified_schema[col] = 'FLOAT'
                        elif unified_schema[col] != 'STRING' and bq_type == 'STRING':
                             unified_schema[col] = 'STRING'
                        # Add more type promotion rules if needed


            except Exception as e:
                print(f"❌ Failed schema inference on {file}: {e}")

        if not unified_schema:
            print("❌ Schema inference failed for META_ADS. Exiting processing for this source.")
            csv_files = [] # Clear files if schema inference fails
        else:
            # Convert unified schema dictionary to BigQuery SchemaField list
            schema = [
                bigquery.SchemaField(re.sub(r'[^a-zA-Z0-9_]', '', k), v) # Ensure schema field names are valid
                for k, v in unified_schema.items()
            ]
            print("✅ Inferred Unified Schema (META_ADS):")
            for s in schema:
                print(f"  {s.name}: {s.field_type}")


    if csv_files and schema: # Only proceed with data processing if files and schema are available
        # Generates a string like '2026_02_26' (Recommended for better sorting)
        current_date_string = datetime.now().strftime('%Y_%m_%d')
        backup_table_name = f"{table_id}_backup_{current_date_string}"
        backup_table_ref = dataset_ref.table(backup_table_name)
        backup_query = f"""
        CREATE TABLE IF NOT EXISTS `{project_id}.{dataset_id}.{backup_table_name}` AS
        SELECT * FROM `{project_id}.{dataset_id}.{table_id}`;
        """
        print(f"\n--- Creating BigQuery Backup Table: {backup_table_name} ---")
        try:
            bq_client.query(backup_query).result()
            print(f"✅ Backup table '{backup_table_name}' created successfully (if it didn't exist) or already exists.")
        except Exception as e:
            print(f"❌ Failed to create backup table '{backup_table_name}': {e}")
            # Decide if processing should halt on backup failure; for now, print error and continue
        # --- End Backup Mechanism ---

        print("\n--- Starting Data Processing and Loading Pass (META_ADS) ---")
        for i, source_file in enumerate(csv_files):
            print(f"\nProcessing: {source_file}")
            try:
                blob = bucket.blob(source_file)
                blob.download_to_filename("/tmp/source.csv")
                df = pd.read_csv("/tmp/source.csv", encoding='utf-8', on_bad_lines='skip', low_memory=False)

                if df.empty:
                    print("⚠️ Empty file. Skipping.")
                    continue

                # Clean column names in the DataFrame to match the schema
                df.columns = [re.sub(r'[^a-zA-Z0-9_ /]', '', col).lower() for col in df.columns]
                df.columns = [col.replace('/', '_') for col in df.columns]
                df.columns = [col.replace(' ', '_') for col in df.columns]


                if 'ad_name' in df.columns:
                    df['ad_name'] = df['ad_name'].astype(str)
                    # Handle potential missing parts more robustly
                    split_result = df['ad_name'].str.split('-', expand=True)
                    df['ad_part1'] = split_result[0].fillna('') if 1 in split_result.columns else ''
                    df['ad_part2'] = split_result[1].fillna('') if 1 in split_result.columns else ''
                    df['ad_part3'] = split_result[2].fillna('') if 2 in split_result.columns else ''

                if 'ad_set_name' in df.columns:
                    split_result = df['ad_set_name'].str.split('_', expand=True)
                    df['set_part1'] = split_result[0].fillna('') if 1 in split_result.columns else ''
                    df['set_part2'] = split_result[1].fillna('') if 1 in split_result.columns else ''
                    df['set_part3'] = split_result[2].fillna('') if 2 in split_result.columns else ''

                if 'campaign_name' in df.columns:
                    split_result = df['campaign_name'].str.split('_', expand=True)
                    df['camp_part1'] = split_result[0].fillna('') if 1 in split_result.columns else ''
                    df['camp_part2'] = split_result[1].fillna('') if 1 in split_result.columns else ''
                    df['camp_part3'] = split_result[2].fillna('') if 2 in split_result.columns else ''

                    def extract_show_title(row):
                        current_show_title = row.get('show_title')

                        # Preserve existing valid show_title if not empty or the default branding
                        if pd.notna(current_show_title) and \
                           current_show_title not in ['', 'AJ360 Branding', 'nan', None]: # Explicitly check for common 'empty' strings
                            return current_show_title

                        # Proceed with lookup only if no valid show_title is found
                        for part in [row.get('ad_part1'), row.get('ad_part2'), row.get('ad_part3'), row.get('set_part3'), row.get('set_part2'), row.get('set_part1'), row.get('camp_part3'), row.get('camp_part2'), row.get('camp_part1')]: # Use .get for safety
                            if part and part in term_lookup:
                                print(f"Term: {part} -> {term_lookup[part]}") # Removed verbose print
                                if term_lookup[part] == "AJ360 Branding":
                                  continue # Keep searching if it's the default branding from lookup
                                else:
                                  return term_lookup[part]
                        print(f"Term: {part} -> None (Defaulting to AJ360 Branding)") # Removed verbose print
                        return "AJ360 Branding"

                    df['show_title'] = df.apply(extract_show_title, axis=1)
                    df.drop(columns=['ad_part1', 'ad_part2', 'ad_part3','set_part1', 'set_part2', 'set_part3', 'camp_part1', 'camp_part2', 'camp_part3'], inplace=True, errors='ignore')
                else:
                    df['show_title'] = None # Ensure show_title column exists even if ad_name is missing

                # Ensure all columns from the unified schema are present and in the correct order
                all_schema_cols = [field.name for field in schema]
                for col in all_schema_cols:
                    if col not in df.columns:
                        df[col] = None # Add missing columns with None

                df = df[all_schema_cols] # Reindex DataFrame to match schema column order


                # Type casting and cleaning based on the unified schema
                for field in schema:
                    col = field.name
                    typ = field.field_type

                    if col in df.columns: # Ensure column exists before processing
                        if typ == 'INTEGER':
                            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
                        elif typ == 'FLOAT':
                            df[col] = pd.to_numeric(df[col], errors='coerce')
                        elif typ == 'BOOLEAN':
                            # Attempt to convert common string representations to boolean
                            df[col] = df[col].astype(str).str.lower().map({'true': True, 'false': False, '1': True, '0': False}).astype('boolean')
                        elif typ == 'TIMESTAMP':
                            # Attempt to parse various date/datetime formats, coerce errors
                            # Convert to string in a format BigQuery expects (YYYY-MM-DD HH:MM:SS)
                            # Handle potential NaT values after conversion
                            df[col] = pd.to_datetime(df[col], errors='coerce')
                            df[col] = df[col].dt.strftime('%Y-%m-%d %H:%M:%S').replace({pd.NA: None, np.nan: None}) # Convert NaT/NaN to None

                        else: # STRING type
                            # More robust cleaning for string columns
                            df[col] = df[col].astype(str).str.replace('"', '', regex=False).str.replace("'", '', regex=False).str.replace('\n', ' ', regex=False).str.replace('\r', ' ', regex=False)
                            df[col] = df[col].replace('nan', None) # Replace 'nan' string with None


                # Save the cleaned DataFrame to a temporary CSV file
                df.to_csv("/tmp/clean.csv", index=False, sep=',', quoting=csv.QUOTE_MINIMAL, encoding='utf-8', header=True)
                blob_clean = bucket.blob(clean_file)
                blob_clean.upload_from_filename("/tmp/clean.csv")
                print("✅ Cleaned file uploaded to GCS.")

                # Set write disposition to always append
                write_disposition = bigquery.WriteDisposition.WRITE_APPEND
                print(f"Setting write disposition to: {write_disposition}")

                # Load data into BigQuery
                load_config = bigquery.LoadJobConfig(
                    source_format=bigquery.SourceFormat.CSV,
                    skip_leading_rows=1,
                    schema=schema, # Use the unified schema
                    write_disposition=write_disposition,
                    encoding='UTF-8',
                    quote_character='"', # Match the quote character used in to_csv
                    field_delimiter=',',
                    allow_quoted_newlines=True, # Allow quoted newlines
                    allow_jagged_rows=True, # Allow jagged rows
                    max_bad_records=1000 # Allow up to 1000 bad records
                )

                destination_uri = f"gs://{bucket_name}/{clean_file}"
                load_job = bq_client.load_table_from_uri(destination_uri, meta_ads_table_ref, job_config=load_config) # Load into META_ADS table
                load_job.result()
                print(f"✅ Loaded {source_file} to BigQuery table {table_id}.")

            except Exception as e:
                print(f"❌ Error processing or loading {source_file}: {e}")
                if hasattr(e, 'errors'):
                    print("BigQuery Load Job Errors:")
                    for error in e.errors:
                        print(error)


    print("\nFinished processing all META_ADS CSV files.")

✅ Google Cloud authenticated successfully.
Listing CSV files in bucket: aj360_data with prefix: META_ADS/meta_ads_performance_5-7Mar2026.csv
✅ Found 1 CSV files to process for META_ADS.
✅ Dataset aj360 exists.
Loading lookup table from BigQuery...


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Loaded lookup table with 699 terms.

--- Schema Inference (META_ADS) ---
Inferring schema from: META_ADS/meta_ads_performance_5-7Mar2026.csv
  campaign_id: int64
  ad_id: int64
  campaign_name: object
  ad_name: object
  platform: object
  day: object
  ad_set_id: int64
  ad_set_name: object
  objective: object
  link_ad_settings: object
  ctr_all: float64
  post_engagements: float64
  registrations_completed: float64
  video_plays: float64
  amount_spent_usd: float64
  content_views: float64
  impressions: int64
  clicks_all: int64
  app_installs: float64
  views: int64
  reach: int64
  outbound_clicks: float64
  outbound_ctr_clickthrough_rate: float64
  ctr_link_clickthrough_rate: float64
  unique_clicks_all: int64
  unique_ctr_all: float64
  unique_outbound_clicks: float64
  unique_link_clicks: float64
  link_clicks: float64
  unique_outbound_ctr_clickthrough_rate: float64
  unique_ctr_link_clickthrough_rate: float64
  reporting_starts: object
  reporting_ends: object
✅ Inferred U